In [28]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from tabulate import tabulate
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight 
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Carico csv

In [29]:
# Carico SOLO i dataset radiomici
df_duke = pd.read_csv(FILE_PATH / "duke_lesions_radiomic_medsam.csv")
df_ambl = pd.read_csv(FILE_PATH / "ambl_lesions_radiomic_medsam.csv")

print("DUKE shape:", df_duke.shape)
print("AMBL shape:", df_ambl.shape)


DUKE shape: (291, 109)
AMBL shape: (82, 111)


# Definisco il target di interesse

In [30]:
target_row = {
    "PR" : ["PR", "PR [SII]"]
}

# Cerco a colonna PR nei csv

In [31]:
def get_target_column (df, target_aliases):
    for col in target_aliases:
        if col in df.columns:
            return col
    raise ValueError(f"Nessuna colonna target trovata tra: {target_aliases}")

# Binarizzo ambl

In [32]:
df_ambl["PR_binario"] = (
        pd.to_numeric(df_ambl["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

# Applico la funzione

In [33]:
target_duke = get_target_column(df_duke, target_row["PR"])
target_ambl = "PR_binario"

print("Target PR DUKE:", target_duke)
print("Target PR AMBL:", target_ambl)

Target PR DUKE: PR
Target PR AMBL: PR_binario


# Switch ruoli

In [34]:
df_train_source = df_ambl      # ora AMBL è training
df_test_external = df_duke     # DUKE è test esterno

target_train = "PR_binario"    # target AMBL (binario)
target_test  = target_duke     # target DUKE


# Controllo delle classi

In [35]:
print("\nDistribuzione PR DUKE:")
print(df_duke[target_duke].value_counts(dropna=False))

print("\nDistribuzione PR AMBL:")
print(df_ambl[target_ambl].value_counts(dropna=False))



Distribuzione PR DUKE:
PR
0    157
1    134
Name: count, dtype: int64

Distribuzione PR AMBL:
PR_binario
0    46
1    36
Name: count, dtype: int64


# Preparo le feature

In [36]:
def prepare_features(df, target_col):
    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast", target_col
    ]

    X = df.drop(columns=features_to_drop, errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    return X

# Split duke (Train e Testing)

In [37]:
# Split AMBL (Train e Validation)
X_full = prepare_features(df_train_source, target_train)
y_full = df_train_source[target_train]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X_full, y_full))

train_original = df_train_source.iloc[train_idx].reset_index(drop=True)
val_internal   = df_train_source.iloc[val_idx].reset_index(drop=True)

print("Train AMBL:", train_original.shape)
print("Val AMBL:", val_internal.shape)


Train AMBL: (65, 112)
Val AMBL: (17, 112)


# Classi uguali (solo in fase di Train)

In [38]:
class_counts = train_original[target_train].value_counts()
n_min = class_counts.min()

train_balanced = (
    train_original
    .groupby(target_train, group_keys=False)
    .apply(lambda x: x.sample(n=n_min, random_state=42))
    .reset_index(drop=True)
)

print("Distribuzione PR dopo bilanciamento:")
print(train_balanced[target_train].value_counts())


Distribuzione PR dopo bilanciamento:
PR_binario
0    29
1    29
Name: count, dtype: int64


/var/folders/fp/7t9yf86d2d38gysngbl33_y40000gn/T/ipykernel_57089/3155790320.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_original


# Feature finali

In [39]:
X_train = prepare_features(train_balanced, target_train)
FEATURE_COLUMNS = X_train.columns.tolist()
y_train = train_balanced[target_train]

X_val = prepare_features(val_internal, target_train)
X_val = X_val.reindex(columns=FEATURE_COLUMNS)

X_test = prepare_features(df_test_external, target_test)
X_test = X_test.reindex(columns=FEATURE_COLUMNS)


# Definisco il modello XGBoost

In [40]:
model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        scale_pos_weight=1,
    )
model.fit(X_train, y_train);

# Testing Duke

In [41]:
y_test = df_test_external[target_test]

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

print("\nDUKE – External Test")
print(f"F1 : {f1_score(y_test, y_test_pred):.3f}")
print(f"AUC: {roc_auc_score(y_test, y_test_proba):.3f}")
print(f"ACC: {accuracy_score(y_test, y_test_pred):.3f}")



DUKE – External Test
F1 : 0.631
AUC: 0.546
ACC: 0.460


# Validazione ambl

In [42]:
y_val = val_internal[target_train]

y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

print("\nAMBL – Validation")
print(f"F1 : {f1_score(y_val, y_val_pred):.3f}")
print(f"AUC: {roc_auc_score(y_val, y_val_proba):.3f}")
print(f"ACC: {accuracy_score(y_val, y_val_pred):.3f}")



AMBL – Validation
F1 : 0.727
AUC: 0.929
ACC: 0.824
